In [1]:
import pandas as pd
import numpy as np
import os
from Bio import Entrez,SeqIO
Entrez.email = "huangyan8@genomics.cn"  # 设置你的邮箱地址
from ete3 import NCBITaxa, PhyloTree
ncbi = NCBITaxa()
from collections import Counter
import warnings
warnings.filterwarnings("ignore")
# 提供一个物种Taxonomy ID的列表
table_path='C:\\Users\\huangyan8\\Desktop\\work\\2024-12-03 TE HT Draft\\Tables\\'
samples=pd.read_excel(table_path+"Table S1\\Table S1.Sample_Information.xlsx").fillna("")
species_list =samples['Taxonomy_ID'].unique() # 人类、小鼠、大鼠的示例
samples['CODE_ID']=samples['Specie_ID']
taxid_dic={}
for i in range(samples.shape[0]):
    taxid_dic[samples.loc[i,'CODE_ID']]=samples.loc[i,'Taxonomy_ID']
samples.head(1)

,Name,Specie_ID,Taxonomy_ID,Assembly_Accession,Assembly,Genome_Source,Plant_Group,subphylum,class,order,family,genus,CODE_ID
0,Acer negundo,PBXX,4023,GCA_025594385.1,ASM2559438v1,NCBI,Dicots,Streptophytina,Magnoliopsida,Sapindales,Sapindaceae,Acer,PBXX


### Step 1. Get TE and TEpep Bed (With orthogroup)

In [144]:
data=pd.read_excel('D:\\19.TE_HT\\Work\\HTTs_Species_Pairs_Pos_Stats_With_SeqCounts.V2.xlsx')
data1=data[data['Group'].apply(lambda x: 1 if x.startswith("TEpep") else 0)==0].reset_index(drop=True)
data2=data[data['Group'].apply(lambda x: 1 if x.startswith("TEpep") else 0)==1].reset_index(drop=True)
print(data1.shape[0],len(list(data1['Group'].unique())))
print(data2.shape[0],len(list(data2['Group'].unique())))

25046 287
24664 399


In [145]:
te_type='TEpep'
ortho_path='D:\\19.TE_HT\\01.TEpep_Ortho\\'
families=['CACTA', 'Copia', 'Gypsy', 'hAT', 'Helitron', 'LINE', 'LTR_Roo', 'Mariner', 'Mutator', 'Other']
bed_path='D:\\18.TE_Evolution\\06.Bed\\TEpep_Bed\\'
ortho_dic={}
for family in families:
    ortho=pd.read_csv(ortho_path+family+"\\Orthogroups\\Orthogroups_V5.tsv",sep='\t').fillna("")
    cols=[col.split(".T")[0] for col in ortho.columns]
    if te_type=='TEpep':
        ortho['Orthogroup']=te_type+'_'+family+"_"+ortho['Orthogroup']
    elif family !='Other':
        if te_type+'_'+family not in ortho.loc[0,"Orthogroup"]:
            ortho['Orthogroup']=te_type+'_'+family+"_"+ortho['Orthogroup']
    ortho.columns=cols
    ortho_dic[family]=ortho

In [146]:
ortho_dic["Gypsy"].head(1)

,Orthogroup,AABQ,ABLZ,AIGJ,AKEL,AKRJ,AOHR,ASLP,AVOY,BBCM,...,ZGPW,ZHMB,ZJMF,ZKTA,ZOHO,ZQTT,ZUZJ,ZVWC,ZVZR,ZZXT
0,TEpep_Gypsy_OG0000000,,"ABLZ#TP11199, ABLZ#TP11408, ABLZ#TP11417, ABLZ...","AIGJ#TP10050, AIGJ#TP10077, AIGJ#TP27164, AIGJ...","AKEL#TP10209, AKEL#TP10810, AKEL#TP108585, AKE...",,"AOHR#TP148286, AOHR#TP152075, AOHR#TP168770, A...",ASLP#TP7741,"AVOY#TP10556, AVOY#TP20282, AVOY#TP29868, AVOY...","BBCM#TP139182, BBCM#TP142014, BBCM#TP177103, B...",...,,"ZHMB#TP4669, ZHMB#TP4686, ZHMB#TP4925, ZHMB#TP...",,,"ZOHO#TP3755, ZOHO#TP3919, ZOHO#TP4334, ZOHO#TP...","ZQTT#TP4465, ZQTT#TP4474, ZQTT#TP5322, ZQTT#TP...",,,"ZVZR#TP12658, ZVZR#TP26483, ZVZR#TP32702, ZVZR...",


In [147]:
HT_Species=data2["HT_Target"].tolist()+data2["HT_Source"].tolist()
HT_Species=list(set(HT_Species))
len(HT_Species)

421

In [149]:
seq_dic={}
for s in HT_Species:
    seq_dic[s]={}
    s_df=data2[(data2['HT_Target']==s)|(data2['HT_Source']==s)]
    for f,s_f_df in s_df.groupby("Family"):
        ortho_df=ortho_dic[f]
        if "TEpep" not in ortho_df.loc[0,'Orthogroup']:
            ortho_df['Orthogroup']="TEpep_"+f+"_"+ortho_df['Orthogroup']
        df1=ortho_df[['Orthogroup',s]]
        df1=df1.set_index("Orthogroup")
        HT_family_groups=s_f_df['Group'].tolist()
        for g in HT_family_groups:
            seq_dic[s][g]=df1.loc[g,s]
df=pd.DataFrame.from_dict(seq_dic,orient='index').fillna("").T.reset_index()

In [150]:
df.to_excel("D:\\19.TE_HT\\Work\\HT_TEpep_Seqs_in_Each_HTpep_Species.xlsx",index=False)
print(df.shape)

(399, 422)


### HT Ortho Families

In [151]:
data1.head(1)

,Family,Group,HT_Target,HT_Source,HT_Sim,Target_Species,Source_Species,Source_Species_id,Target_Species_id,Source_Seq_Count,Target_Seq_Count
0,Gypsy,TE_Gypsy_OG0021796,BUJI,FVXP,0.999645,Taxus chinensis,Oryza sativa DG,4530,29808,1639,6050


In [152]:
HT_Species=data1["HT_Target"].tolist()+data1["HT_Source"].tolist()
HT_Species=list(set(HT_Species))
len(HT_Species)

417

In [153]:
ortho_dic={}
for family,f_df in data1.groupby("Family"):
    ortho=pd.read_csv("D:\\19.TE_HT\\03.TE_Ortho\\"+family+"\\Orthogroups\\Orthogroups_V5.tsv",sep='\t')
    cols=[col.split(".T")[0] for col in ortho.columns]
    ortho.columns=cols
    ortho_dic[family]=ortho

In [154]:
for family,f_df in data1.groupby("Family"):
    ortho_df=ortho_dic[family]
    print(ortho_df.loc[0,'Orthogroup'])

TE_CACTA_OG0000000
TE_Copia_OG0000000
TE_Gypsy_OG0000000
TE_Helitron_OG0000000
TE_Mariner_OG0000000
TE_Mutator_OG0000000
TE_Solo_OG0000000
TE_PIF-Harbinger_OG0000000
TE_hAT_OG0000000


In [155]:
seq_dic={}
for s in HT_Species:
    seq_dic[s]={}
    s_df=data1[(data1['HT_Target']==s)|(data1['HT_Source']==s)]
    for f,s_f_df in s_df.groupby("Family"):
        ortho_df=ortho_dic[f]
        df1=ortho_df[['Orthogroup',s]]
        df1=df1.set_index("Orthogroup")
        HT_family_groups=s_f_df['Group'].tolist()
        for g in HT_family_groups:
            seq_dic[s][g]=df1.loc[g,s]
df=pd.DataFrame.from_dict(seq_dic,orient='index').fillna("").T.reset_index()
df.head()

,index,BGQP,AIGJ,VQED,STTP,VUQT,IRCT,NGLJ,BUKK,GJNO,...,LOUF,NDIU,ASLP,UWLT,MTGG,NGPU,MFXD,BYSY,JEQQ,SDJN
0,TE_Unknown_OG0002620,"BGQP#TE930, BGQP#TE929, BGQP#TE1677, BGQP#TE16...","AIGJ#TE567741, AIGJ#TE5210, AIGJ#TE455986, AIG...","VQED#TE814036, VQED#TE769529, VQED#TE769527, V...","STTP#TE2790, STTP#TE271, STTP#TE2357, STTP#TE2...","VUQT#TE287, VUQT#TE905, VUQT#TE772, VUQT#TE362...","IRCT#TE915, IRCT#TE76, IRCT#TE291, IRCT#TE1677...","NGLJ#TE97074, NGLJ#TE86512, NGLJ#TE8546, NGLJ#...","BUKK#TE784, BUKK#TE256, BUKK#TE1704, BUKK#TE1070","GJNO#TE963, GJNO#TE817, GJNO#TE486, GJNO#TE20,...",...,,,,,,,,,,
1,TE_Copia_OG0000008,,"AIGJ#TE303, AIGJ#TE5740, AIGJ#TE79000, AIGJ#TE...","VQED#TE4733, VQED#TE5172, VQED#TE18184, VQED#T...",,,,"NGLJ#TE5080, NGLJ#TE5082, NGLJ#TE5698, NGLJ#TE...",,,...,,,,,,,,,,
2,TE_Copia_OG0000109,,,,,,,,,,...,,,,,,,,,,
3,TE_Copia_OG0017853,,,,,,,"NGLJ#TE3171, NGLJ#TE20496, NGLJ#TE20519, NGLJ#...",,,...,,,,,,,,,,
4,TE_Gypsy_OG0015099,,,,,,,,,,...,,,,,,,,,,


In [156]:
df.to_excel("D:\\19.TE_HT\\Work\\HT_TE_Seqs_in_Each_HTpep_Species.xlsx",index=False)
print(df.shape)

(287, 418)


In [53]:
for col in df.columns[1:]:
    df[col]=df[col].apply(lambda x:len(x.split(", ")) if x !='' else 0)
df.head()

,index,EZSP,KAFX,ZHCR,DLVS,NGLJ,PNCF,WDCV,NJEL,BUJI,...,JEQQ,LOUF,OYYK,BTJS,STTP,RNSE,DCPK,BVFO,ITWP,XDOS
0,TE_CACTA_OG0000134,40,636,35,25,27,111,75,32,38,...,0,0,0,0,0,0,0,0,0,0
1,TE_Copia_OG0000109,70,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,TE_Copia_OG0000270,66,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,TE_Copia_OG0000378,28,0,0,0,0,0,109,0,106,...,0,0,0,0,0,0,0,0,0,0
4,TE_Copia_OG0004274,52,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [54]:
df.to_excel(table_path+"HT_TE_SeqsCounts_in_Each_Sample.xlsx",index=False)
print(df.shape)

(622, 211)


### Step 2.Get Co-HT families
- TE-TE
- TE-TEpep
- TEpep-TEpep

In [171]:
TEpep=pd.read_excel("D:\\19.TE_HT\\Work\\HT_TEpep_Seqs_in_Each_HTpep_Species.xlsx").fillna('')
TE=pd.read_excel("D:\\19.TE_HT\\Work\\HT_TE_Seqs_in_Each_HTpep_Species.xlsx").fillna('')   
print(TEpep.shape,TE.shape)

(399, 422) (287, 418)


In [172]:
TE_species=TE.columns[1:]
TEpep_species=TEpep.columns[1:]
overlap=set(TE_species).intersection(TEpep_species)
len(overlap)

356

In [173]:
for s in TE_species:
    if s not in overlap:
        te=pd.read_csv("D:\\19.TE_HT\\05.HT_Bed\\TE\\"+s+"_EDTA.bed",sep='\t')
        ht_seqs={}
        for seq in ", ".join(TE[TE[s]!=''][s].tolist()).split(", "):
            ht_seqs[seq.strip()]=1
        te=te[te['Seq_ID'].apply(lambda x:1 if x in ht_seqs else 0)==1]
        te.to_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\'+s+"_TE.bed",sep='\t',index=False)

In [174]:
for s in TEpep_species:
    if s not in overlap:
        tepep=pd.read_csv("D:\\19.TE_HT\\05.HT_Bed\\TEpep\\"+s+"_TPSI.bed",sep='\t')
        ht_seqs={}
        for seq in ", ".join(TEpep[TEpep[s]!=''][s].tolist()).split(", "):
            ht_seqs[seq.strip()]=1
        tepep=tepep[tepep["Seq_ID"].apply(lambda x:1 if x in ht_seqs else 0)==1]
        tepep.to_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_TEpep\\'+s+"_TEpep.bed",sep='\t',index=False)

In [ ]:
for s in overlap:
    te=pd.read_csv("D:\\19.TE_HT\\05.HT_Bed\\TE\\"+s+"_EDTA.bed",sep='\t')
    tepep=pd.read_csv("D:\\19.TE_HT\\05.HT_Bed\\TEpep\\"+s+"_TPSI.bed",sep='\t')
    ht_seqs={}
    for seq in ", ".join(TE[TE[s]!=''][s].tolist()).split(", "):
        ht_seqs[seq.strip()]=1
    for seq in ", ".join(TEpep[TEpep[s]!=''][s].tolist()).split(", "):
        ht_seqs[seq.strip()]=1
    te=te[te['Seq_ID'].apply(lambda x:1 if x in ht_seqs else 0)==1]
    tepep=tepep[tepep["Seq_ID"].apply(lambda x:1 if x in ht_seqs else 0)==1]
    print(s, te.shape,tepep.shape)
    tepep=tepep.reset_index(drop=True)
    tepep['Overlap']=0
    ol=0
    for i in range(tepep.shape[0]):
        chrom=tepep.loc[i,'#CHROM']
        start=tepep.loc[i,'Start']
        end=tepep.loc[i,'End']
        loc_te=te[(te['#CHROM']==chrom)&(te['Start']>=start-10000)&(te['End']<=end+10000)]
        if loc_te.shape[0]>0:
            #print(s,tepep.loc[i,'TE_Type'])
            ol+=1
            tepep.loc[i,'Overlap']=1
            
    te=te.reset_index(drop=True)
    te['Overlap']=0
    for i in range(te.shape[0]):
        chrom=te.loc[i,'#CHROM']
        start=te.loc[i,'Start']
        end=te.loc[i,'End']
        loc_tepep=tepep[(tepep['#CHROM']==chrom)&(tepep['Start']>=start-10000)&(tepep['End']<=end+10000)]
        if loc_tepep.shape[0]>0:
            #print(s,te.loc[i,'TE_Type'])
            ol+=1
            te.loc[i,'Overlap']=1
    te.to_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\'+s+"_TE.bed",sep='\t',index=False)
    tepep.to_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_TEpep\\'+s+"_TEpep.bed",sep='\t',index=False)
    print(s,ol)

### Gene and ncRNA

In [176]:
#data1=pd.read_csv("D:\\19.TE_HT\\Work\\HTT_TEpep_Species_Pairs.csv")
family='Protein'
data1=pd.read_csv("D:\\19.TE_HT\\Work\\HTT_"+family+"_Species_Pairs.V3.csv")
#data1['Group']="TEpep"+"_"+data1['Family']+"_"+data1['Group']
data1=data1[data1['HT_Sim']>0.8]
HT_Species=data1["HT_Target"].tolist()+data1["HT_Source"].tolist()
HT_Species=list(set(HT_Species))
len(HT_Species)

358

In [177]:
ortho_df=pd.read_csv("D:\\19.TE_HT\\02.Protein_Ortho\\Orthogroups_V2.tsv",sep='\t')
ortho_df.head(1)

,Orthogroup,PBXX,QIBM,BUKK,GJNO,WWFJ,ZVZR,XNAW,XHDS,VMNX,...,QVQN,JZZF,FDFI,BUJI,TXXW,VQRB,TXNO,RYGG,JVNO,MQHU
0,FLOR_0,"KAI9152718.1, KAI9152720.1, KAI9152753.1, KAI9...","KAK1549029.1, KAK1549034.1, KAK1549189.1, KAK1...","KAK1281762.1, KAK1281794.1, KAK1281797.1, KAK1...","KAK1256598.1, KAK1257428.1, KAK1257503.1, KAK1...","CEY00_Acc00113, CEY00_Acc00120, CEY00_Acc00911...","XP_057459234.1, XP_057460223.1, XP_057460509.1...","KAI5054010.1, KAI5055026.1, KAI5055687.1, KAI5...","AET1Gv20131800, AET1Gv20381900, AET1Gv20579400...","AH000230-RA, AH000316-RA, AH000518-RA, AH00052...",...,"KAK9082848.1, KAK9083177.1, KAK9084093.1, KAK9...","KAK9084585.1, KAK9084665.1, KAK9084762.1, KAK9...","KAK9081675.1, KAK9086250.1, KAK9086594.1, KAK9...","KAH9287570.1, KAH9288488.1, KAH9289006.1, KAH9...","KAK4740675.1, KAK4741144.1, KAK4741198.1, KAK4...","KAK4762442.1, KAK4762717.1, KAK4762756.1, KAK4...","CAL4870422.1, CAL4870423.1, CAL4870546.1, CAL4...","CAI8582917.1, CAI8583969.1, CAI8584439.1, CAI8...","WVY89330.1, WVY89570.1, WVY90007.1, WVY90016.1...","KAJ9670667.1, KAJ9670668.1, KAJ9670669.1, KAJ9..."


In [178]:
seq_dic={}
for s in HT_Species:
    seq_dic[s]={}
    s_df=data1[(data1['HT_Target']==s)|(data1['HT_Source']==s)]
    df1=ortho_df[['Orthogroup',s]]
    df1=df1.set_index("Orthogroup")
    HT_family_groups=s_df['Group'].tolist()
    for g in HT_family_groups:
        seq_dic[s][g]=df1.loc[g,s]
        
df=pd.DataFrame.from_dict(seq_dic,orient='index').T.reset_index()
df=df.fillna("")
df.to_excel("D:\\19.TE_HT\\Work\\HT_Protein_Seqs_in_Each_HT_Species.xlsx",index=False)

In [179]:
df.head(1)

,index,BGQP,XCFZ,QZAA,SAQA,PRMX,KPNG,EZSP,LSLZ,GPTC,...,RVKF,DEJD,TZNK,REPT,MBFQ,RMKG,ZJMF,MTTE,RYGG,NNNE
0,K02983,"XP_034695492.1, XP_034700069.1, XP_034700154.1",Zm00040ab299930_T001,XP_030541765.1,GWHPAAEV019444.1,"Zm00031ab292550_T001, Zm00031ab389950_T001","Vitvi09g04055, Vitvi11g04055","gene-LSAT_5X146160, gene-LSAT_7X38741, gene-LS...","Zm00022ab194530_T001, Zm00022ab262410_T001, Zm...",GWHPACEX016322,...,,,,,,,,,,


In [180]:
group_dic={}
for i in range(df.shape[0]):
    group=df.loc[i,'index']
    for s in HT_Species:
        if df.loc[i,s]!='':
            for seq in df.loc[i,s].split(", "):
                group_dic[seq]=group

In [181]:
df=df.fillna('')
for s in HT_Species:
    gene=pd.read_csv("D:\\19.TE_HT\\05.HT_Bed\\Gene\\"+s+"_Gene.bed",sep='\t')
    ht_seqs={}
    for seq in ", ".join(df[df[s]!=''][s].tolist()).split(", "):
        ht_seqs[seq.strip()]=1
    gene=gene[gene["Gene"].apply(lambda x:1 if x in ht_seqs else 0)==1]
    gene['HT_Group']=gene['Gene'].apply(lambda x:group_dic[x])
    gene.to_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_Gene\\'+s+"_Gene.bed",sep='\t',index=False)
    #print(gene.shape,len(list(ht_seqs.keys())))

In [ ]:
for s in HT_Species:
    gene=pd.read_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_Gene\\'+s+"_Gene.bed",sep='\t')
    gene['Overlap']=''
    if s+"_TEpep.bed" in os.listdir("D:\\19.TE_HT\\05.HT_Bed\\HT_TEpep\\"):
        tepep=pd.read_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_TEpep\\'+s+"_TEpep.bed",sep='\t')
        for i in range(gene.shape[0]):
            chrom=gene.loc[i,'#CHROM']
            start=gene.loc[i,'Start']
            end=gene.loc[i,'End']
            loc_tepep=tepep[(tepep['#CHROM']==chrom)&(tepep['Start']>=start-10000)&(tepep['End']<=end+10000)]
            if loc_tepep.shape[0]>0:
                gene.loc[i,'Overlap']+='TEpep;'
                #print(s," TEpep")
    if s+"_TE.bed" in os.listdir("D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\"):
        te=pd.read_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\'+s+"_TE.bed",sep='\t')
        for i in range(gene.shape[0]):
            chrom=gene.loc[i,'#CHROM']
            start=gene.loc[i,'Start']
            end=gene.loc[i,'End']
            loc_te=te[(te['#CHROM']==chrom)&(te['Start']>=start-10000)&(te['End']<=end+10000)]
            if loc_te.shape[0]>0:
                gene.loc[i,'Overlap']+='TE;'
                #print(s," TE")
    gene.to_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_Gene\\'+s+"_Gene.bed",sep='\t',index=False)

In [ ]:
files=os.listdir('D:\\19.TE_HT\\05.HT_Bed\\HT_TEpep\\')
for f in files:
    #print(f)
    # TEpep是筛选有overlap的gene
    R=[]
    s=f.split("_")[0]
    te=pd.read_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_TEpep\\'+f,sep='\t')
    te['Overlap_Gene']=''
    gene=pd.read_csv("D:\\19.TE_HT\\05.HT_Bed\\Gene\\"+s+"_Gene.bed",sep='\t').fillna("")
    for i in range(te.shape[0]):
        chrom=te.loc[i,'#CHROM']
        start=te.loc[i,'Start']
        end=te.loc[i,'End']
        loc_gene1=gene[(gene['#CHROM']==chrom)&(gene['Start']>=start)&(gene['Start']<=end)]
        loc_gene2=gene[(gene['#CHROM']==chrom)&(gene['End']>=start)&(gene['End']<=end)]
        loc_gene3=gene[(gene['#CHROM']==chrom)&(gene['Start']<=start)&(gene['End']>=end)]
        loc_gene=pd.concat([loc_gene1,loc_gene2,loc_gene3]).drop_duplicates()
        if loc_gene.shape[0]>0:
            loc_gene['Overlap']='TEpep'
            #print(s,te.loc[i,'TE_Type']," | ".join(loc_gene['Gene'].tolist()))
            te.loc[i,'Overlap_Gene']=" | ".join([str(gene) for gene in loc_gene['Gene'].tolist()])
            R.append(loc_gene)
    if R!=[]:
        R=pd.concat(R)
        R=R.drop_duplicates()
        R.to_csv("D:\\19.TE_HT\\05.HT_Bed\\HT_Gene\\Overlap\\"+s+"_Gene_TEpep.bed",sep='\t',index=False)
    te.to_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_TEpep\\'+f,sep='\t',index=False)

In [186]:
for f in os.listdir('D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\'):
     # TE是筛选有gene 位于TE 上下游10Kb的基因
    R=[]
    s=f.split("_")[0]
    te=pd.read_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\'+f,sep='\t')
    te['Overlap_Gene']=''
    gene=pd.read_csv("D:\\19.TE_HT\\05.HT_Bed\\Gene\\"+s+"_Gene.bed",sep='\t').fillna("")
    for i in range(te.shape[0]):
        chrom=te.loc[i,'#CHROM']
        start=te.loc[i,'Start']
        end=te.loc[i,'End']
        loc_gene=gene[(gene['#CHROM']==chrom)&(gene['Start']>=start-10000)&(gene['End']<=end+10000)&(gene['Gene']!='')]
        if loc_gene.shape[0]>0:
            loc_gene['Overlap']='TE'
            #print(s,te.loc[i,'TE_Type']," | ".join(loc_gene['Gene'].tolist()))
            te.loc[i,'Overlap_Gene']=" | ".join([str(gene) for gene in loc_gene['Gene'].tolist()])
            R.append(loc_gene)
    if R!=[]:
        R=pd.concat(R)
        R=R.drop_duplicates()
        R.to_csv("D:\\19.TE_HT\\05.HT_Bed\\HT_Gene\\Overlap\\"+s+"_Gene_TE.bed",sep='\t',index=False)
    te.to_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\'+f,sep='\t',index=False)

### HTG Genes

In [ ]:
save_path='D:\\19.TE_HT\\08.HGT\\'
for i in range(samples.shape[0]):
    if i%20==0:
        print(i)
    sample=samples.loc[i,'Specie_ID']
    gene=pd.read_csv("D:\\19.TE_HT\\05.HT_Bed\\Gene\\"+sample+"_Gene.bed",sep='\t').fillna("")
    te=[]
    if sample+"_TE.bed" in os.listdir('D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\'):
        te1=pd.read_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\'+sample+"_TE.bed",sep='\t').fillna("")
        te.append(te1)
    if sample+"_TEpep.bed" in os.listdir('D:\\19.TE_HT\\05.HT_Bed\\HT_TEpep\\'):
        te2=pd.read_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_TEpep\\'+sample+"_TEpep.bed",sep='\t').fillna("")
        te.append(te2)
    if te!=[]:
        te=pd.concat(te).sort_values(['#CHROM','Start'])
        gene_dic={}
        for chrom,g_df in gene.groupby("#CHROM"):
            gene_dic[chrom]=g_df
        r=[]
        for chrom,c_df in te.groupby("#CHROM"):
            c_df=c_df.reset_index(drop=True)
            if chrom in gene_dic:
                g_df=gene_dic[chrom]
                c_df['Overlap_Gene']=''
                for j in range(c_df.shape[0]):
                    start=c_df.loc[j,'Start']
                    end=c_df.loc[j,'End']
                    loc_gene1=g_df[(g_df['Start']>=start)&(g_df['Start']<=end)]
                    loc_gene2=g_df[(g_df['End']>=start)&(g_df['End']<=end)]
                    loc_gene3=g_df[(g_df['Start']<=start)&(g_df['End']>=end)]
                    loc_gene=pd.concat([loc_gene1,loc_gene2,loc_gene3]).drop_duplicates()
                    if loc_gene.shape[0]>0:
                        c_df.loc[j,'Overlap_Gene']=" | ".join([str(gene) for gene in loc_gene['Gene'].tolist()])
                r.append(c_df)
        if r!=[]:
            r=pd.concat(r)
            r=r[r['Overlap_Gene']!='']
            if r.shape[0]>0:
                print(sample,r.shape[0])
                r.to_csv(save_path+sample+"_HGT.csv",index=False)

In [ ]:
for f in os.listdir('D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\'):
    # TE是筛选有gene 位于TE 上下游10Kb的基因
    R=[]
    s=f.split("_")[0]
    te=pd.read_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\'+f,sep='\t')
    te['Overlap_Gene']=''
    gene=pd.read_csv("D:\\19.TE_HT\\05.HT_Bed\\Gene\\"+s+"_Gene.bed",sep='\t').fillna("")
    for i in range(te.shape[0]):
        chrom=te.loc[i,'#CHROM']
        start=te.loc[i,'Start']
        end=te.loc[i,'End']
        loc_gene1=gene[(gene['#CHROM']==chrom)&(gene['Start']>=start)&(gene['Start']<=end)]
        loc_gene2=gene[(gene['#CHROM']==chrom)&(gene['End']>=start)&(gene['End']<=end)]
        loc_gene=pd.concat([loc_gene1,loc_gene2])
        if loc_gene.shape[0]>0:
            loc_gene['Overlap']='TEpep'
            #print(s,te.loc[i,'TE_Type']," | ".join(loc_gene['Gene'].tolist()))
            te.loc[i,'Overlap_Gene']=" | ".join([str(gene) for gene in loc_gene['Gene'].tolist()])
    te.to_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\'+f,sep='\t',index=False)

In [187]:
#data1=pd.read_csv("D:\\19.TE_HT\\Work\\HTT_TEpep_Species_Pairs.csv")
family='Protein'
data1=pd.read_csv("D:\\19.TE_HT\\Work\\HTT_"+family+"_Species_Pairs.V3.csv")
#data1['Group']="TEpep"+"_"+data1['Family']+"_"+data1['Group']
data1=data1[(data1['HT_Sim']>0.8)&(data1['Group'].apply(lambda x:1 if x.startswith("RF") else 0)==1)]
print(data1.shape)
data1.head()

(178, 7)


,Family,Group,HT_Target,HT_Source,HT_Sim,Target_Species,Source_Species
1561748,Protein,RF00906,YSDX,BUKK,0.812565,Beta vulgaris,Acorus calamus
1561749,Protein,RF00906,YSDX,BUJI,0.803658,Beta vulgaris,Taxus chinensis
1561894,Protein,RF00906,DXPQ,BUKK,0.846408,Dipteronia dyeriana,Acorus calamus
1561895,Protein,RF00906,DXPQ,BUJI,0.842190,Dipteronia dyeriana,Taxus chinensis
1561896,Protein,RF00906,DXPQ,YIRO,0.826654,Dipteronia dyeriana,Lactuca saligna


In [188]:
HT_Species=data1["HT_Target"].tolist()+data1["HT_Source"].tolist()
HT_Species=list(set(HT_Species))
len(HT_Species)

54

In [189]:
seq_dic={}
for s in HT_Species:
    seq_dic[s]={}
    s_df=data1[(data1['HT_Target']==s)|(data1['HT_Source']==s)]
    df1=ortho_df[['Orthogroup',s]]
    df1=df1.set_index("Orthogroup")
    HT_family_groups=s_df['Group'].tolist()
    for g in HT_family_groups:
        seq_dic[s][g]=df1.loc[g,s]
        
df=pd.DataFrame.from_dict(seq_dic,orient='index').T.reset_index()
df.to_excel("D:\\19.TE_HT\\Work\\HT_ncRNA_Seqs_in_Each_HT_Species.xlsx",index=False)

In [190]:
df=df.fillna('')
group_dic={}
for i in range(df.shape[0]):
    group=df.loc[i,'index']
    for s in HT_Species:
        if df.loc[i,s]!='':
            for seq in df.loc[i,s].split(", "):
                group_dic[seq]=group
for s in HT_Species:
    ncRNA=pd.read_csv("D:\\19.TE_HT\\05.HT_Bed\\ncRNA\\"+s+"_ncRNA.Bed",sep='\t')
    ht_seqs={}
    for seq in ", ".join(df[df[s]!=''][s].tolist()).split(", "):
        ht_seqs[seq.strip()]=1
    ncRNA=ncRNA[ncRNA["ncRNA"].apply(lambda x:1 if x in ht_seqs else 0)==1]
    ncRNA['Group']=ncRNA['ncRNA'].apply(lambda x:group_dic[x])
    ncRNA.to_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_ncRNA\\'+s+"_ncRNA.bed",sep='\t',index=False)

In [ ]:
for s in HT_Species:
    ncRNA=pd.read_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_ncRNA\\'+s+"_ncRNA.bed",sep='\t')
    ncRNA['Overlap']=''
    if s+"_TEpep.bed" in os.listdir("D:\\19.TE_HT\\05.HT_Bed\\HT_TEpep\\"):
        tepep=pd.read_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_TEpep\\'+s+"_TEpep.bed",sep='\t')
        for i in range(ncRNA.shape[0]):
            chrom=ncRNA.loc[i,'#CHROM']
            start=ncRNA.loc[i,'Start']
            end=ncRNA.loc[i,'End']
            loc_tepep=tepep[(tepep['#CHROM']==chrom)&(tepep['Start']>=start-10000)&(tepep['End']<=end+10000)]
            if loc_tepep.shape[0]>0:
                ncRNA.loc[i,'Overlap']+='TEpep;'
                print(s," TEpep")
    #if s+"_TE.bed" in os.listdir("D:\\19.TE_HT\\05.HT_Bed\\TE\\"):
        #te=pd.read_csv('D:\\19.TE_HT\\05.HT_Bed\\TE\\'+s+"_TE.bed",sep='\t')
    te=pd.read_csv("D:\\19.TE_HT\\05.HT_Bed\\TE\\"+s+"_EDTA.bed",sep='\t')
    for i in range(ncRNA.shape[0]):
        chrom=ncRNA.loc[i,'#CHROM']
        start=ncRNA.loc[i,'Start']
        end=ncRNA.loc[i,'End']
        loc_te=te[(te['#CHROM']==chrom)&(te['Start']>=start-100)&(te['End']<=end+100)]
        if loc_te.shape[0]>0:
            ncRNA.loc[i,'Overlap']+='TE;'
            print(s," TE")
    ncRNA.to_csv('D:\\19.TE_HT\\05.HT_Bed\\HT_ncRNA\\'+s+"_ncRNA.bed",sep='\t',index=False)

### Step 3. Co-HT Network

In [23]:
HT_TE_families=list(data1['Group'].unique())
HT_Tepep_families=list(data2['Group'].unique())
HT_TE_path='D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\'
HT_TEpep_path='D:\\19.TE_HT\\05.HT_Bed\\HT_TEpep\\'
files=os.listdir(HT_TE_path)
for file in files:
    sample=file.split("_")[0]
    te=pd.read_csv(HT_TE_path+file,sep='\t').fillna("")
    te=te.drop_duplicates()
    te.to_csv(HT_TE_path+file,sep='\t',index=False)
    Source=[]
    Target=[]
    Distance=[]
    Co_Local_Type=[]
    for chrom,c_df in te.groupby("#CHROM"):
        #print(chrom)
        #c_df=c_df.drop_duplicates()
        if c_df.shape[0]>1:
            c_df=c_df.sort_values("Start").reset_index(drop=True)
            c_df['Start_Shift']=c_df['Start'].shift(-1)
            c_df['Distance']=c_df['Start_Shift']-c_df['End']
            c_df['Next_Orthogroup']=c_df['Orthogroup'].shift(-1)
            c_df=c_df[(c_df['Distance']<10000)&(c_df['Distance']>-10000)&(c_df['Orthogroup']!=c_df['Next_Orthogroup'])].reset_index(drop=True)# 
            for i in range(c_df.shape[0]):
                Source.append(c_df.loc[i,'Orthogroup'])
                Target.append(c_df.loc[i,'Next_Orthogroup'])
                Distance.append(c_df.loc[i,'Distance'])
                Co_Local_Type.append("TE-TE")
    #print("TE done!\t",len(Source))
    if 'Overlap' not in te.columns:
        te['Overlap']=0
    te=te[te['Overlap']==1].reset_index(drop=True)
    if te.shape[0]>0:
        tepep=pd.read_csv(HT_TEpep_path+file.replace("TE","TEpep"),sep='\t').fillna("")
        tepep=tepep.drop_duplicates()
        for j in range(te.shape[0]):
            chrom=te.loc[j,'#CHROM']
            start=te.loc[j,'Start']
            end=te.loc[j,'End']
            temp=tepep[tepep['#CHROM']==chrom]
            temp=temp[((temp['Start']>=(start-10000))&(temp['Start']<=(end+10000)))|
                     ((temp['End']>=(start-10000))&(temp['End']<=(end+10000)))].reset_index(drop=True)
            if temp.shape[0]>0:
                for a in range(temp.shape[0]):
                    tepep_group=temp.loc[a,'Orthogroup']
                    if temp.loc[a,'End']<=start:
                        distance=start-temp.loc[a,'End']
                    elif temp.loc[a,'Start']>=end:
                        distance=temp.loc[a,'Start']-end
                    elif (temp.loc[a,'Start']<end)&(temp.loc[a,'Start']>start):
                        distance=temp.loc[a,'Start']-end
                    elif (temp.loc[a,'End']>=start)&(temp.loc[a,'End']<=end):
                        distance=temp.loc[a,'End']-start
                    distance=min([abs(temp.loc[a,'Start']-start),abs(temp.loc[a,'End']-start),\
                                  abs(temp.loc[a,'Start']-end),abs(temp.loc[a,'End']-end)])
                    Source.append(te.loc[j,'Orthogroup'])
                    Target.append(tepep_group)
                    Distance.append(distance)
                    Co_Local_Type.append("TE-TEpep")
    #print("TE-TEpep done!\t",len(Source))
    if file.replace("TE",'TEpep') in os.listdir(HT_TEpep_path):
        tepep=pd.read_csv(HT_TEpep_path+file.replace("TE","TEpep"),sep='\t').fillna("")
        tepep=tepep.drop_duplicates()
        tepep.to_csv(HT_TEpep_path+file.replace("TE","TEpep"),sep='\t',index=False)
        for chrom,c_df in tepep.groupby("#CHROM"):
            #c_df=c_df.drop_duplicates()
            if c_df.shape[0]>1:
                c_df=c_df.sort_values("Start").reset_index(drop=True)
                c_df['Start_Shift']=c_df['Start'].shift(-1)
                c_df['Distance']=c_df['Start_Shift']-c_df['End']
                c_df['Next_Orthogroup']=c_df['Orthogroup'].shift(-1)
                c_df=c_df[(c_df['Distance']<10000)&(c_df['Distance']>-10000)\
                          &(c_df['Orthogroup']!=c_df['Next_Orthogroup'])].reset_index(drop=True)#
                if c_df.shape[0]>0:
                    for i in range(c_df.shape[0]):
                        Source.append(c_df.loc[i,'Orthogroup'])
                        Target.append(c_df.loc[i,'Next_Orthogroup'])
                        Distance.append(c_df.loc[i,'Distance'])
                        Co_Local_Type.append("TEpep-TEpep")
    #print("TEpep done!\t",len(Source))
    if len(Source)>0:
        data=pd.DataFrame(Source,columns=['Source'])
        data['Target']=pd.DataFrame(Target)
        data['Distance']=pd.DataFrame(Distance)
        data['Co_Local_Type']=pd.DataFrame(Co_Local_Type)
        data['Sample']=sample
        data.to_excel('D:\\19.TE_HT\\05.HT_Bed\\Co_HT\\'+sample+"_Co_HT_Network.xlsx",index=False)
        print(sample,data['Co_Local_Type'].value_counts())

In [ ]:
HT_TE_families=list(data1['Group'].unique())
HT_Tepep_families=list(data2['Group'].unique())
HT_TE_path='D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\'
HT_TEpep_path='D:\\19.TE_HT\\05.HT_Bed\\HT_TEpep\\'
files=os.listdir('D:\\19.TE_HT\\05.HT_Bed\\Co_HT\\')
for file in files:
    sample=file.split("_Co_")[0]
    data=pd.read_excel('D:\\19.TE_HT\\05.HT_Bed\\Co_HT\\'+sample+"_Co_HT_Network.xlsx")
    use_groups=list(set(data['Source'].tolist()+data['Target'].tolist()))
    r=[]
    if sample+"_TE.bed" in os.listdir(HT_TE_path):
        te=pd.read_csv(HT_TE_path+sample+"_TE.bed",sep='\t').fillna("")
        r.append(te)
    if sample+"_TEpep.bed" in os.listdir(HT_TEpep_path):
        tepep=pd.read_csv(HT_TEpep_path+sample+"_TEpep.bed",sep='\t').fillna("")
        r.append(tepep)
    r=pd.concat(r)
    r=r[r['Orthogroup'].apply(lambda x:1 if x in use_groups else 0)==1]
    r=r.sort_values(['#CHROM','Start']).reset_index(drop=True)
    r['Orthogroup_Shift1']=r['Orthogroup'].shift(-1)
    r['Orthogroup_Shift2']=r['Orthogroup'].shift(1)
    r=r[(r['Overlap_Gene']!='')&((r['Orthogroup']!=r['Orthogroup_Shift1'])|(r['Orthogroup']!=r['Orthogroup_Shift2']))]
    if r.shape[0]>0:
        co_HT_genes=list(set(" | ".join(r['Overlap_Gene'].tolist()).split(" | ")))
        print(sample,len(co_HT_genes))
        if sample in rice_samples:
            co_HT_genes=get_rice_gene_names(co_HT_genes)
        if sample in maize_samples:
            co_HT_genes=get_maize_gene_names(co_HT_genes)
        with open('D:\\19.TE_HT\\07.Enrichment\\7.Co_HT_Genes\\'+sample+"_Co_HT.txt",'w') as f0:
            for g in co_HT_genes:
                f0.write(g+"\n")
            f0.close()

In [ ]:
save_path='D:\\19.TE_HT\\05.HT_Bed\\Co_HT\\'
files=os.listdir(save_path)
R=[]
for file in files:
    df=pd.read_excel(save_path+file)
    R.append(df)
R=pd.concat(R).reset_index(drop=True)
print(R.shape)
R['Pair']=R['Source']+":"+R['Target']
def get_sorted_pair(x):
    x=x.split(":")
    x.sort()
    return ":".join(x)
R=R[R['Pair'].isnull()==0]
R['Pair']=R['Pair'].apply(get_sorted_pair)
R['Taxon_ID']=R['Sample'].apply(lambda x:taxid_dic[x])
df0=pd.DataFrame(R.groupby(["Pair",'Co_Local_Type'])['Distance'].mean()).reset_index()
print(df0.shape)
df0.head()

In [ ]:
df0['Seqs_Count']=0
df0['Samples']=''
df0['Species']=''
df0['Species_Count']=0
for i in range(df0.shape[0]):
    if i%1000==0:
        print(i)
    pair=df0.loc[i,'Pair']
    r=R[R['Pair']==pair]
    df0.loc[i,'Seqs_Count']=r.shape[0]
    species=list(r['Taxon_ID'].unique())
    df0.loc[i,'Samples']=" | ".join(list(r['Sample'].unique()))
    df0.loc[i,'Species']=" | ".join([str(s) for s in species])
    df0.loc[i,'Species_Count']=len(species)
df0=df0[df0['Species_Count']>1]
print(df0.shape)

In [ ]:
df0['Source']=df0['Pair'].apply(lambda x:x.split(":")[0])
df0['Target']=df0['Pair'].apply(lambda x:x.split(":")[1])
df0.to_excel(table_path+"Table S7//Table S7. Co-localization Relations of TE families in HTTs.xlsx",index=False)

### Step 4. Co-HT Network Top100

In [24]:
HT_TE_families=list(data1['Group'].unique())
HT_Tepep_families=list(data2['Group'].unique())
HT_TE_path='D:\\19.TE_HT\\05.HT_Bed\\HT_TE\\'
HT_TEpep_path='D:\\19.TE_HT\\05.HT_Bed\\HT_TEpep\\'
files=os.listdir('D:\\19.TE_HT\\05.HT_Bed\\Co_HT\\')
data=pd.read_excel(table_path+"Table S7\\Table S7. Co-localization Relations of TE families in HTTs.xlsx")
from collections import Counter
c=Counter(data['Source'].tolist()+data['Target'].tolist())
groups=[]
for k,v in c.most_common(100):
    groups.append(k)
len(groups)

In [2]:
table_path='C:\\Users\\huangyan8\\Desktop\\work\\2024-12-03 TE HT Draft\\Tables\\'
df0=pd.read_excel(table_path+"Table S7//Table S7. Co-localization Relations of TE families in HTTs.xlsx")
df0.head()

,Co-HT_Family_Pair,Co_Local_Type,Median Distance of Seqs(bp),Seqs_Count,Samples,Species,Species_Count,Source,Target
0,TE_CACTA_OG0000079:TE_CACTA_OG0000083,TE-TE,3378.484211,190,BUJI | FDFI | PNCF | ZHCR | ZVWC,29808 | 152371 | 2759587 | 4283 | 3352,5,TE_CACTA_OG0000079,TE_CACTA_OG0000083
1,TE_CACTA_OG0000079:TE_CACTA_OG0000230,TE-TE,1061.117647,17,AIXU | KAFX,212925 | 94328,2,TE_CACTA_OG0000079,TE_CACTA_OG0000230
2,TE_CACTA_OG0000079:TE_CACTA_OG0000439,TE-TE,2047.541667,24,AIXU | KAFX,212925 | 94328,2,TE_CACTA_OG0000079,TE_CACTA_OG0000439
3,TE_CACTA_OG0000079:TE_Copia_OG0000000,TE-TE,924.045455,22,BUJI | KAFX,29808 | 94328,2,TE_CACTA_OG0000079,TE_Copia_OG0000000
4,TE_CACTA_OG0000079:TE_Copia_OG0000008,TE-TE,3524.818182,11,KAFX | PNCF | ZHCR,94328 | 2759587 | 4283,3,TE_CACTA_OG0000079,TE_Copia_OG0000008


In [3]:
df0['Source']=df0['Co-HT_Family_Pair'].apply(lambda x:x.split(":")[0])
df0['Target']=df0['Co-HT_Family_Pair'].apply(lambda x:x.split(":")[1])